In [1]:
suppressPackageStartupMessages({
  library(data.table); library(dplyr); library(ggplot2); library(stringr); library(ggtext) 
})

## Plots grouped by NES Up vs Down 

In [2]:
fig_path <- '../../../data/rna/pseudobulk/results/fgsea_dotplots'
dir.create(fig_path, recursive = TRUE, showWarnings = FALSE)


files <- Sys.glob('../../../data/rna/pseudobulk/results/fgsea_results/*.csv')

In [3]:
if (length(files) > 0) {
  for (f in files) {
    message('Processing: ', f)

    # --- read data ---
    gsea_res <- fread(f)

    # tag database + pathway
    gsea_res <- gsea_res %>%
      mutate(
        db_tag = case_when(
          str_detect(gmt_db, regex('MSigDB', ignore_case = TRUE)) ~ 'MSigDB',
          str_detect(gmt_db, regex('Reactome', ignore_case = TRUE)) ~ 'Reactome',
          str_detect(gmt_db, regex('ENCODE', ignore_case = TRUE)) ~ 'ENCODE',
          str_detect(gmt_db, regex('KEGG', ignore_case = TRUE)) ~ 'KEGG',
          TRUE ~ gmt_db
        ),
        pathway = paste0(db_tag, ' | ', pathway),
        comparison = f %>%
          basename() %>%
          tools::file_path_sans_ext() %>%
          str_remove('_fgsea$') %>%
          str_remove('_gsea$') %>%
          str_replace('l2', 'L2 ') %>%
          str_replace('^bmmc_', 'BMMC ') %>%
          str_replace('^pbmc_', 'PBMC ') %>%
          str_replace('_', ' ')
      )

    es_col <- 'NES'
    TOP_K <- 50

    gsea_res <- gsea_res %>%
      mutate(
          celltype = factor(
              stringr::str_to_title(gsub('_', ' ', celltype)),
              levels = stringr::str_to_title(gsub('_', ' ', levels(factor(celltype))))
          )
      )

    df <- gsea_res %>%
      mutate(
        padj_lt_0.05 = padj < 0.05,
        arm_label = factor(comparison),
        x_axis = factor(celltype)
      )

    # --- top pathways ---
    top_by_arm <- df %>%
      filter(padj_lt_0.05) %>%
      group_by(arm_label, pathway) %>%
      summarise(
        best_padj = min(padj, na.rm = TRUE),
        best_abs_nes = max(abs(.data[[es_col]]), na.rm = TRUE),
        score = best_abs_nes * -log10(best_padj),
        .groups = 'drop'
      ) %>%
      group_by(arm_label) %>%
      slice_max(order_by = score, n = TOP_K, with_ties = FALSE) %>%
      ungroup()

    keep_set <- unique(top_by_arm$pathway)

    plot_df <- df %>%
      filter(pathway %in% keep_set) %>%
      group_by(pathway) %>%
      mutate(direction = case_when(
        all(na.omit(sign(.data[[es_col]][padj_lt_0.05])) == 1) ~ 'Up',
        all(na.omit(sign(.data[[es_col]][padj_lt_0.05])) == -1) ~ 'Dn',
        TRUE ~ 'Mix'
      )) %>%
      ungroup() %>%
      # Improved pathway name formatting
      mutate(
        # First shorten the pathway name by removing redundant words
        pathway_short = pathway %>%
          str_remove_all("(?i)\\b(pathway|signaling|signal|response|regulation|process|activity|binding|complex)\\b") %>%
          str_remove_all("(?i)\\b(GO:|KEGG:|REACTOME:|WP_)\\b") %>%
          str_replace_all("_", " ") %>%
          str_squish(), # removes extra whitespace
        
        # Then wrap to multiple lines with shorter width
        pathway_lab = str_wrap(pathway_short, width = 35)
      )

    ord <- plot_df %>%
      group_by(pathway_lab) %>%
      summarise(med_nes = median(.data[[es_col]], na.rm = TRUE), .groups = 'drop') %>%
      arrange(med_nes) %>%
      pull(pathway_lab)

    plot_df <- plot_df %>%
      mutate(
        pathway_lab = factor(pathway_lab, levels = ord),
        # Make database tag bold (before the |)
        pathway_lab = str_replace(pathway_lab, '^(.*?) \\|', '**\\1** |')
      )

    g_all_comp <- ggplot(plot_df, aes(x_axis, pathway_lab)) +
      geom_point(
        shape = 21, stroke = 0.6,
        aes(
          fill  = .data[[es_col]],
          size  = pmax(1e-6, -log10(padj)),
          alpha = padj_lt_0.05
        ),
        color = 'black'
      ) +
      scale_fill_gradient2(
        name = 'NES', low = 'blue4', mid = 'white',
        high = 'red', midpoint = 0
      ) +
      scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.35), guide = 'none') +
      scale_size(
        range = c(1.6, 6), breaks = c(3, 6, 9, 12, 15, 18),
        name = expression(-log[10](padj))
      ) +
      facet_grid(direction ~ arm_label, space = 'free_y', scales = 'free_y') +
      labs(x = NULL, y = NULL) +
      theme_bw(base_size = 14) +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, face = "bold"),
        axis.text.y = element_markdown(size = 10), # slightly smaller text
        strip.text = element_text(size = 11),
        panel.grid.minor = element_blank(),
        panel.spacing.x = unit(0.4, 'lines'),
        legend.key.height = unit(0.5, 'cm')
      )

    out_file <- file.path(fig_path, paste0(basename(tools::file_path_sans_ext(f)), '_dotplot.png'))
    ggsave(out_file, g_all_comp, width = 16, height = 16, dpi = 300)
    message('Saved: ', out_file)
  }
} else {
  message('No DEG files found. Skipping.')
}

Processing: results/fgsea_results/fgsea_results_bmmc_ASCT1y_vs_ASCT2y.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_ASCT1y_vs_ASCT2y_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_ASCT1y_vs_Healthy.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_ASCT1y_vs_Healthy_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_ASCT2y_vs_Healthy.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_ASCT2y_vs_Healthy_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_ASCT90d_vs_ASCT1y.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_ASCT90d_vs_ASCT1y_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_ASCT90d_vs_Healthy.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_ASCT90d_vs_Healthy_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_EI_vs_ASCT1y.csv

Saved: results/fgsea_dotplots/fgsea_results_bmmc_EI_vs_ASCT1y_dotplot.png

Processing: results/fgsea_results/fgsea_results_bmmc_EI_vs_ASCT2y.csv

Sav